# Text Extraction

we need to create a dictionnary 

key : n° volume, n° article

value : content (title and substract)

In [1]:
import nltk
import pandas as pd
from bs4 import BeautifulSoup
import requests
import os
import time


volume number is inside : #main > div > div > ul > li:nth-child(1) > h2 > span:nth-child(1)

 issue number is inside : #main > div > div > ul > li:nth-child(1) > ul > li:nth-child(1) > a

 href of issue containing articles : #main > div > div > ul > li:nth-child(1) > ul > li:nth-child(1) > a['href']

 link to article : #main > div > div > div > section > ol > li:nth-child(1) > article > div.app-card-open__main > h3 > a
 
 function that can extract title and abstract

In [2]:
# link to extract from
link = "https://link.springer.com/journal/12065/volumes-and-issues"

# the dictionnay will be containing :
# key   : volume number, article number
# value : content of the article (title and absract)
articles_dict = {}


In [3]:
BASE_URL = "https://link.springer.com"
MAIN_URL = f"{BASE_URL}/journal/12065/volumes-and-issues"

def extract_all_articles(save_dir="articles_data"):
    os.makedirs(save_dir, exist_ok=True)
    articles_dict = {}

    # Step 1️⃣: Load main page with all volumes
    main_html = requests.get(MAIN_URL).text
    main_soup = BeautifulSoup(main_html, "html.parser")

    # Each volume = one <li> under #main > div > div > ul
    volume_items = main_soup.select("#main > div > div > ul > li")

    print(f"📚 Found {len(volume_items)} volumes.")

    for v_index, volume_item in enumerate(volume_items, start=1):
        # Extract volume number
        volume_span = volume_item.select_one("h2 > span:nth-child(1)")
        if not volume_span:
            continue
        volume_number = volume_span.get_text(strip=True)
        print(f"\n🔵 Processing {volume_number} ({v_index}/{len(volume_items)})")

        articles_dict[volume_number] = {}

        # Find all issues under this volume
        issue_links = volume_item.select("ul > li > a")
        print(f"  Found {len(issue_links)} issues in {volume_number}.")

        for issue_index, issue_a in enumerate(issue_links, start=1):
            issue_number = issue_a.get_text(strip=True)
            issue_href = issue_a.get("href")
            issue_url = f"{BASE_URL}{issue_href}"
            print(f"  🟢 Issue {issue_number} ({issue_index}/{len(issue_links)}) → {issue_url}")

            # Step 2️⃣: Visit issue page
            try:
                issue_html = requests.get(issue_url).text
                issue_soup = BeautifulSoup(issue_html, "html.parser")
            except Exception as e:
                print(f"  ⚠️ Error loading issue page: {e}")
                continue

            # Step 3️⃣: Extract all article links
            article_links = issue_soup.find_all("a", href=lambda x: x and x.startswith("/article/"))
            print(f"    Found {len(article_links)} articles in {issue_number}.")

            issue_articles = []

            for i, link in enumerate(article_links, start=1):
                href = link.get("href")
                title = link.get_text(strip=True)

                # Skip empty or invalid titles
                if not title:
                    continue

                article_url = f"{BASE_URL}{href}"
                print(f"    🔹 Article {i}: {title}")

                try:
                    article_html = requests.get(article_url).text
                    article_soup = BeautifulSoup(article_html, "html.parser")

                    abstract_div = article_soup.find("div", {"class": "c-article-section__content"})
                    abstract = abstract_div.find("p").get_text(strip=True) if abstract_div else "Abstract not found"

                    issue_articles.append({
                        "title": title,
                        "abstract": abstract
                    })

                    # Save to file
                    file_name = f"{volume_number.replace(' ', '_')}_{issue_number.replace(' ', '_')}_article_{i}.txt"
                    file_path = os.path.join(save_dir, file_name)
                    with open(file_path, "w", encoding="utf-8") as f:
                        f.write(f"Title: {title}\n\nAbstract:\n{abstract}\n")

                except Exception as e:
                    print(f"    ⚠️ Error fetching article: {e}")

                # polite delay
                time.sleep(2)

            # Step 4️⃣: Store results in the dictionary
            articles_dict[volume_number][issue_number] = issue_articles

    print("\n🎉 Extraction complete! All data saved in dictionary and text files.")
    return articles_dict


In [ ]:
data = extract_all_articles() 

📚 Found 18 volumes.

🔵 Processing Volume 18 (1/18)
  Found 6 issues in Volume 18.
  🟢 Issue Issue 6 (1/6) → https://link.springer.com/journal/12065/volumes-and-issues/18-6
    Found 4 articles in Issue 6.
    🔹 Article 1: Solution of frequency deviation and multi objective probabilistic optimal power flow of transmission network incorporating UPFC controller using driving training based optimization
    🔹 Article 2: Improved two-archive evolutionary algorithm for constrained multi-objective optimization of WWTP based on intergenerational information guidance
    🔹 Article 3: Privacy-preserving data aggregation in WBNAs using neuro-evolutionary algorithms and post-quantum homomorphic encryption
    🔹 Article 4: An evolutionary algorithm tailored to the quadratic assignment problem
  🟢 Issue Issue 5 (2/6) → https://link.springer.com/journal/12065/volumes-and-issues/18-5
    Found 21 articles in Issue 5.
    🔹 Article 1: A comprehensive review of optimization approaches of district coolin

In [ ]:
# saving the data
import pickle
with open("data/articles_dict.pkl", "wb") as f:
    pickle.dump(data, f)


# Text Display part 2

Define ways to display
- either by article : give number of article and volume
- either by volume : dipsplay all articles

In [ ]:
import pickle

# Path to your pickle file
file_path = "data/articles_dict.pkl"  # replace with actual filename

# Load the data
with open(file_path, "rb") as f:
    data = pickle.load(f)

print("✅ Data loaded successfully!")
print(type(data))        # check data type (should be dict)
print(len(data))         # number of volumes (for example)

#  displaying the strucutre of data
print(list(data.keys()))  # show the first few keys (usually volumes)
print(list(data.values())[:5])


✅ Data loaded successfully!
<class 'dict'>
18
['Volume 18', 'Volume 17', 'Volume 16', 'Volume 15', 'Volume 14', 'Volume 13', 'Volume 12', 'Volume 11', 'Volume 10', 'Volume 9', 'Volume 8', 'Volume 7', 'Volume 6', 'Volume 5', 'Volume 4', 'Volume 3', 'Volume 2', 'Volume 1']
[{'Issue 6': [{'title': 'Solution of frequency deviation and multi objective probabilistic optimal power flow of transmission network incorporating UPFC controller using driving training based optimization', 'abstract': 'The optimal power flow (OPF) solution for the IEEE 30-bus system is being achieved while taking thermal power (TP) sources into account. In this study, the OPF solution of conventional power systems incorporating FACTS devices with frequency security constraint is investigated. By incorporating FACTS devices into the existing power systems the proposed work improves the power system’s capacity to transfer power and hence reduce generation costs. The objectives are to reduce generation costs and emissio

In [ ]:
#  eliminating informations in the dictionnary where retrival stopped 
def clean_incomplete_data(data):
    cleaned_data = {}

    for volume, issues in data.items():
        valid_issues = {}

        for issue, articles in issues.items():
            # Keep only articles that have both a title and abstract
            valid_articles = [
                art for art in articles
                if isinstance(art, dict)
                and art.get("title")
                and art.get("abstract")
                and art["abstract"].lower() != "abstract not found"
            ]

            # Keep the issue only if it has valid articles
            if valid_articles:
                valid_issues[issue] = valid_articles

        # Keep the volume only if it has valid issues
        if valid_issues:
            cleaned_data[volume] = valid_issues

    print(f"✅ Cleaned data: {len(cleaned_data)} valid volumes remaining.")
    return cleaned_data


data_cleaned = clean_incomplete_data(data)
import pickle
with open("data/cleaned_articles_dict.pkl", "wb") as f:
    pickle.dump(data_cleaned, f)


✅ Cleaned data: 3 valid volumes remaining.


In [ ]:
import pickle

# Path to your pickle file
file_path = "data/cleaned_articles_dict.pkl"  # replace with actual filename

# Load the data
with open(file_path, "rb") as f:
    data = pickle.load(f)

print("✅ Data loaded successfully!")
print(type(data))        # check data type (should be dict)
print(len(data))         # number of volumes (for example)

#  displaying the strucutre of data
print(list(data.keys()))  # show the first few keys (usually volumes)
print(list(data.values())[:5])


✅ Data loaded successfully!
<class 'dict'>
3
['Volume 18', 'Volume 17', 'Volume 16']
[{'Issue 6': [{'title': 'Solution of frequency deviation and multi objective probabilistic optimal power flow of transmission network incorporating UPFC controller using driving training based optimization', 'abstract': 'The optimal power flow (OPF) solution for the IEEE 30-bus system is being achieved while taking thermal power (TP) sources into account. In this study, the OPF solution of conventional power systems incorporating FACTS devices with frequency security constraint is investigated. By incorporating FACTS devices into the existing power systems the proposed work improves the power system’s capacity to transfer power and hence reduce generation costs. The objectives are to reduce generation costs and emissions. In order to illustrate that generation and consumption are in balance, the system frequency must be kept within a safe range. Therefore, in addition to ensuring the lowest producing c

In [ ]:
def display_articles(data, option=""):
    if option == "display_by_volume":
        volume = input("Enter volume name (e.g. Volume 62): ").strip()

        if volume not in data:
            print(f"❌ Volume '{volume}' not found.")
            return

        print(f"\n📘 Displaying all articles from {volume}:")

        for issue, articles in data[volume].items():
            print(f"\n=== {issue} ({len(articles)} articles) ===")
            for i, art in enumerate(articles, start=1):
                print(f"\n🔹 Article {i}: {art['title']}")
                print(f"📝 Abstract:\n{art['abstract']}\n{'-'*60}")

    elif option == "display_by_article":
        volume = input("Enter volume name (e.g. Volume 62): ").strip()
        issue = input("Enter issue name (e.g. Issue 1): ").strip()
        article_num = int(input("Enter article number: ").strip())

        if volume not in data:
            print(f"❌ Volume '{volume}' not found.")
            return
        if issue not in data[volume]:
            print(f"❌ Issue '{issue}' not found in {volume}.")
            return
        if article_num < 1 or article_num > len(data[volume][issue]):
            print(f"❌ Invalid article number for {issue} in {volume}.")
            return

        article = data[volume][issue][article_num - 1]
        print(f"\n📘 {volume} → {issue} → Article {article_num}")
        print(f"\n🔹 Title: {article['title']}")
        print(f"📝 Abstract:\n{article['abstract']}\n")

    else:
        print("⚠️ Invalid option. Use 'display_by_volume' or 'display_by_article'.")


In [ ]:
display_articles(data, option="display_by_volume")



📘 Displaying all articles from Volume 18:

=== Issue 6 (4 articles) ===

🔹 Article 1: Solution of frequency deviation and multi objective probabilistic optimal power flow of transmission network incorporating UPFC controller using driving training based optimization
📝 Abstract:
The optimal power flow (OPF) solution for the IEEE 30-bus system is being achieved while taking thermal power (TP) sources into account. In this study, the OPF solution of conventional power systems incorporating FACTS devices with frequency security constraint is investigated. By incorporating FACTS devices into the existing power systems the proposed work improves the power system’s capacity to transfer power and hence reduce generation costs. The objectives are to reduce generation costs and emissions. In order to illustrate that generation and consumption are in balance, the system frequency must be kept within a safe range. Therefore, in addition to ensuring the lowest producing cost under operating condit

In [ ]:
display_articles(data, option="display_by_article")



📘 Volume 18 → Issue 1 → Article 1

🔹 Title: Stochastic PFRCosSim layer for solving filter redundancy problem in CNNs applied on plant disease classification
📝 Abstract:
Convolutional neural networks (CNNs) have achieved remarkable success in various artificial intelligence domains, particularly in pattern recognition, image processing, and speech recognition. However, the growing complexity of these models introduces challenges related to parameter redundancy, significantly impacting CNN performance. This paper addresses the issue of increasing parameter redundancy, focusing on the specific problem of filter redundancy during CNN training. The proposed approach involves regularization of the initialization filters to reduce redundancy at each convolutional layer. A novel layer, PFRCosSim, is introduced, computing the Cosine similarity between filters used in CNN training to ensure filter homogeneity. we reset the filters using an Orthogonal initialization based on the random choice of

In [ ]:
# option = "" either display a specific article or a all articles of a specific volume
# case option:
#   "display_by_volume:
#     volume = input
#     #  access all articles of volume 
#     content = data where volume_number = volume
#     display content (title and abstract)
#   "display_by_article":
#     article = input, volume = input
#     content = data where n°volume and n°article match
#     display content

# Part 3 Text Preprocessing

In [ ]:
import pickle

# Path to your pickle file
file_path = "data/cleaned_articles_dict.pkl"  # replace with actual filename

# Load the data
with open(file_path, "rb") as f:
    data = pickle.load(f)

print("✅ Data loaded successfully!")
print(type(data))        # check data type (should be dict)
print(len(data))         # number of volumes (for example)

#  displaying the strucutre of data
print(list(data.keys()))  # show the first few keys (usually volumes)
print(list(data.values())[:5])


✅ Data loaded successfully!
<class 'dict'>
3
['Volume 18', 'Volume 17', 'Volume 16']
[{'Issue 6': [{'title': 'Solution of frequency deviation and multi objective probabilistic optimal power flow of transmission network incorporating UPFC controller using driving training based optimization', 'abstract': 'The optimal power flow (OPF) solution for the IEEE 30-bus system is being achieved while taking thermal power (TP) sources into account. In this study, the OPF solution of conventional power systems incorporating FACTS devices with frequency security constraint is investigated. By incorporating FACTS devices into the existing power systems the proposed work improves the power system’s capacity to transfer power and hence reduce generation costs. The objectives are to reduce generation costs and emissions. In order to illustrate that generation and consumption are in balance, the system frequency must be kept within a safe range. Therefore, in addition to ensuring the lowest producing c

In [ ]:

import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\moous\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\moous\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
# for each article extract tokens and add as a item in the dictionnary from data
def tokenize_text(text):
    """Clean and tokenize text into lowercase words without stopwords or punctuation."""
    tokens = word_tokenize(text.lower())
    tokens = [re.sub(r'[^a-z0-9]+', '', t) for t in tokens]  # remove punctuation
    tokens = [t for t in tokens if t and t not in stopwords.words('english')]
    return tokens


def add_tokens_to_data(data):
    """Add a 'tokens' field to each article (based on title + abstract)."""
    for volume, issues in data.items():
        for issue, articles in issues.items():
            for article in articles:
                combined_text = f"{article.get('title', '')} {article.get('abstract', '')}"
                article['tokens'] = tokenize_text(combined_text)
    return data


In [ ]:
data_with_tokens = add_tokens_to_data(data)
volume = list(data.keys())[0]
issue = list(data[volume].keys())[0]
print(data[volume][issue][0]['tokens'])


['solution', 'frequency', 'deviation', 'multi', 'objective', 'probabilistic', 'optimal', 'power', 'flow', 'transmission', 'network', 'incorporating', 'upfc', 'controller', 'using', 'driving', 'training', 'based', 'optimization', 'optimal', 'power', 'flow', 'opf', 'solution', 'ieee', '30bus', 'system', 'achieved', 'taking', 'thermal', 'power', 'tp', 'sources', 'account', 'study', 'opf', 'solution', 'conventional', 'power', 'systems', 'incorporating', 'facts', 'devices', 'frequency', 'security', 'constraint', 'investigated', 'incorporating', 'facts', 'devices', 'existing', 'power', 'systems', 'proposed', 'work', 'improves', 'power', 'system', 'capacity', 'transfer', 'power', 'hence', 'reduce', 'generation', 'costs', 'objectives', 'reduce', 'generation', 'costs', 'emissions', 'order', 'illustrate', 'generation', 'consumption', 'balance', 'system', 'frequency', 'must', 'kept', 'within', 'safe', 'range', 'therefore', 'addition', 'ensuring', 'lowest', 'producing', 'cost', 'operating', 'condi

the vision of the dictionnary:

volume:
    issue:
        article:
            title: ".."
            abstract: ".."
            tokens: [...]
            normalised tokens snow: [...]
            normalised tokens porter: [...]
            normalised tokens lancaster: [...]


In [ ]:
# for each article and for all its tokens, apply stemming 
# and add the results into another item in the dictionnay
# the stemmer will be different : snow, porte stemmer and lancaster


# Part 4: N-Gram Language Model

In [ ]:
# three models are going to be deployed : unigram, bigram, trigram
# for each article
#   for each model, access tokens threw the dictionnary*
#     create a folder of the model
#     for each token generate its frequency and probability 
#       add the results to a new dictionnary article_modeltype containing token as a key and (frequency, probability) as the value
#      save the dictiornnay to the folder 